# Chunking Strategy Comparison

This notebook compares three chunking strategies implemented in the Clinical RAG Platform:
- **Recursive Character Chunker** (default)
- **Semantic Chunker**
- **Sliding Window Chunker**

We analyse chunk size distributions, overlap, and coverage across a sample clinical corpus.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from langchain_core.documents import Document

from src.ingestion.chunker import (
    RecursiveCharacterChunker,
    SemanticChunker,
    SlidingWindowChunker,
    get_chunker,
)

print('Imports successful')

In [ ]:
# Sample clinical documents for comparison
CLINICAL_TEXTS = [
    """Metformin is the cornerstone of pharmacological treatment for type 2 diabetes mellitus (T2DM).
    It belongs to the biguanide class and works primarily by reducing hepatic glucose production 
    through activation of AMP-activated protein kinase (AMPK). Additional mechanisms include 
    improved peripheral insulin sensitivity and reduced intestinal glucose absorption.
    
    Dosing: Start at 500 mg twice daily with meals. Titrate by 500 mg weekly to a maximum of 
    2000-2550 mg/day. Extended-release formulations (XR) improve GI tolerability.
    
    Contraindications: eGFR < 30 mL/min/1.73m². Temporarily discontinue before contrast media 
    administration and resume 48 hours after if renal function is stable.""",
    
    """Acute coronary syndrome (ACS) encompasses a spectrum of conditions: unstable angina (UA), 
    non-ST-elevation myocardial infarction (NSTEMI), and ST-elevation myocardial infarction (STEMI).
    
    The pathophysiology involves rupture of an atherosclerotic plaque with subsequent thrombus 
    formation, leading to partial or complete coronary artery occlusion. Key biomarkers include 
    troponin I and T, which are highly sensitive and specific for myocardial necrosis.
    
    Initial management: Aspirin 300 mg loading dose, oxygen if SpO2 < 94%, IV access, 
    12-lead ECG within 10 minutes of presentation. STEMI requires primary PCI within 90 minutes 
    of first medical contact (door-to-balloon time).""",
    
    """Sepsis is defined as life-threatening organ dysfunction caused by a dysregulated host 
    response to infection (Sepsis-3 definition, 2016). The Sequential Organ Failure Assessment 
    (SOFA) score is used to quantify organ dysfunction. A SOFA score increase of ≥2 points 
    identifies sepsis in a patient with suspected infection.
    
    The 'Sepsis Hour Bundle' (Hour-1): Measure lactate; obtain blood cultures before antibiotics;
    administer broad-spectrum antibiotics; administer 30 mL/kg crystalloid for hypotension or
    lactate ≥ 4 mmol/L; apply vasopressors if hypotension persists after resuscitation."""
]

documents = [
    Document(
        page_content=text,
        metadata={'source': f'clinical_doc_{i}.pdf', 'page_number': 1, 'doc_id': f'doc-{i:04d}'}
    )
    for i, text in enumerate(CLINICAL_TEXTS)
]

print(f'Loaded {len(documents)} clinical documents')
for i, doc in enumerate(documents):
    print(f'  Doc {i}: {len(doc.page_content)} chars')

In [ ]:
# Run all three chunkers
chunkers = {
    'recursive': RecursiveCharacterChunker(chunk_size_tokens=128, chunk_overlap_tokens=16),
    'sliding_window': SlidingWindowChunker(window_tokens=64, stride_tokens=32),
}

results = {}
for name, chunker in chunkers.items():
    chunks = chunker.chunk(documents)
    results[name] = chunks
    print(f'{name}: {len(chunks)} chunks')

In [ ]:
# Compute chunk size statistics
stats = {}
for name, chunks in results.items():
    sizes = [len(c.page_content) for c in chunks]
    # Approximate token count (4 chars per token)
    token_sizes = [s // 4 for s in sizes]
    stats[name] = {
        'count': len(chunks),
        'mean_chars': np.mean(sizes),
        'median_chars': np.median(sizes),
        'std_chars': np.std(sizes),
        'min_chars': np.min(sizes),
        'max_chars': np.max(sizes),
        'mean_tokens': np.mean(token_sizes),
        'sizes': sizes,
    }

print(f'{'Strategy':<18} {'Count':>6} {'Mean tokens':>12} {'Std tokens':>10} {'Max tokens':>10}')
print('-' * 60)
for name, s in stats.items():
    print(f"{name:<18} {s['count']:>6} {s['mean_tokens']:>12.1f} {s['std_chars']/4:>10.1f} {s['max_chars']/4:>10.1f}")

In [ ]:
# Visualise chunk size distributions
fig, axes = plt.subplots(1, len(results), figsize=(14, 5), sharey=False)

colors = {'recursive': '#2196F3', 'sliding_window': '#FF9800'}

for ax, (name, chunks) in zip(axes, results.items()):
    sizes_tokens = [len(c.page_content) // 4 for c in chunks]
    ax.hist(sizes_tokens, bins=15, color=colors.get(name, '#9C27B0'), alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(sizes_tokens), color='red', linestyle='--', label=f'Mean: {np.mean(sizes_tokens):.0f}')
    ax.set_title(f'{name.replace("_", " ").title()}\n({len(chunks)} chunks)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Chunk Size (approx. tokens)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Chunk Size Distributions by Chunking Strategy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/chunk_size_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to docs/chunk_size_distributions.png')

In [ ]:
# Inspect sample chunks from each strategy
for name, chunks in results.items():
    print(f'\n{"="*60}')
    print(f'Strategy: {name.upper()}')
    print(f'Total chunks: {len(chunks)}')
    print(f'Sample chunk (index 0):')
    print(f'  Length: {len(chunks[0].page_content)} chars')
    print(f'  Content: {chunks[0].page_content[:200]}...')
    print(f'  Metadata keys: {list(chunks[0].metadata.keys())}')

In [ ]:
# Coverage analysis: verify all source text is represented
def compute_coverage(original_docs, chunks):
    """Estimate fraction of source text covered in chunks."""
    original_text = ' '.join(d.page_content for d in original_docs)
    combined_chunks = ' '.join(c.page_content for c in chunks)
    
    # Sample 50 random 20-char windows from original
    import random
    random.seed(42)
    n_samples = 50
    window_size = 20
    found = 0
    for _ in range(n_samples):
        if len(original_text) <= window_size:
            break
        start = random.randint(0, len(original_text) - window_size)
        window = original_text[start:start + window_size]
        if window in combined_chunks:
            found += 1
    return found / n_samples

print('Coverage Analysis (fraction of source text present in chunks):')
print('-' * 50)
for name, chunks in results.items():
    coverage = compute_coverage(documents, chunks)
    print(f'{name:<20}: {coverage:.1%}')

In [ ]:
# Overlap analysis for recursive chunker
recursive_chunks = results['recursive']

def measure_overlap(chunks, window=2):
    """Measure average word overlap between consecutive chunks."""
    overlaps = []
    for i in range(len(chunks) - window):
        words_a = set(chunks[i].page_content.lower().split())
        words_b = set(chunks[i + window].page_content.lower().split())
        if words_a and words_b:
            jaccard = len(words_a & words_b) / len(words_a | words_b)
            overlaps.append(jaccard)
    return overlaps

for name, chunks in results.items():
    if len(chunks) > 1:
        overlaps = measure_overlap(chunks)
        if overlaps:
            print(f'{name}: mean Jaccard overlap = {np.mean(overlaps):.3f} (n={len(overlaps)} pairs)')
        else:
            print(f'{name}: insufficient chunks for overlap analysis')

In [ ]:
# Summary comparison bar chart
strategies = list(stats.keys())
mean_tokens = [stats[s]['mean_tokens'] for s in strategies]
counts = [stats[s]['count'] for s in strategies]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

bar_colors = ['#2196F3', '#FF9800']
bars1 = ax1.bar(strategies, mean_tokens, color=bar_colors, alpha=0.85, edgecolor='white')
ax1.set_title('Mean Chunk Size (tokens)', fontweight='bold')
ax1.set_ylabel('Approx. tokens')
ax1.set_ylim(0, max(mean_tokens) * 1.3)
for bar, val in zip(bars1, mean_tokens):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.0f}', ha='center', fontweight='bold')

bars2 = ax2.bar(strategies, counts, color=bar_colors, alpha=0.85, edgecolor='white')
ax2.set_title('Total Chunks Produced', fontweight='bold')
ax2.set_ylabel('Number of chunks')
ax2.set_ylim(0, max(counts) * 1.3)
for bar, val in zip(bars2, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(val), ha='center', fontweight='bold')

for ax in (ax1, ax2):
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels([s.replace('_', '\n') for s in strategies])

plt.suptitle('Chunking Strategy Comparison Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/chunking_strategy_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print final summary
print('\n=== CHUNKING STRATEGY COMPARISON SUMMARY ===')
print(f'\nInput: {len(documents)} documents, {sum(len(d.page_content) for d in documents)} total chars')
print(f'\n{"Strategy":<20} {"Chunks":>8} {"Mean tokens":>12} {"Std tokens":>11}')
print('-' * 55)
for name, s in stats.items():
    print(f"{name:<20} {s['count']:>8} {s['mean_tokens']:>12.1f} {s['std_chars']/4:>11.1f}")
print('\nRecommendation: Recursive chunker provides best balance of')
print('chunk size consistency and clinical text boundary preservation.')